In [ ]:
# tamserve BoxTracker: YOLOX-Nano on the CPU, gray frames, many cameras, one shared model.
# One cell: install, make the 1-channel model, check it is exact, track a panning camera, score with MCC,
# and measure CPU and memory against one process per camera. Runs on Colab's CPU runtime; no GPU, no PyTorch used.
import hashlib, json, os, subprocess, sys, urllib.request
from pathlib import Path

BRANCH = "cpu-boxes"
WORK = Path("/content" if Path("/content").exists() else Path.cwd()) / "boxes"
WORK.mkdir(parents=True, exist_ok=True)
REPO = WORK / "tamserve"
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, "https://github.com/dhaneshkk/tamserve", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openvino>=2024.0", "onnx", "opencv-python-headless", "psutil"], check=True)
sys.path.insert(0, str(REPO / "src"))
os.environ["PYTHONPATH"] = str(REPO / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")

import cv2, numpy as np, openvino as ov
from tamserve.boxes import BoxTracker, Detector, gray_model, _letterbox

def fetch(url, sha256):
    path = WORK / url.rsplit("/", 1)[1]
    if not path.exists():
        data = urllib.request.urlopen(url, timeout=120).read()
        assert hashlib.sha256(data).hexdigest() == sha256, f"{url} is not the pinned file"
        path.write_bytes(data)
    return path

nano = fetch("https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.onnx",
             "c789161ed43c8269fcd4e67c67eeeb4e80c622da2eb296a20bc6007bd18a0b7d")
dog = fetch("https://raw.githubusercontent.com/Megvii-BaseDetection/YOLOX/6ddff4824372906469a7fae2dc3206c7aa4bbaee/assets/dog.jpg",
            "5a9522051c3cec2bbd2f6323fccba32e8fbf3ddcc2b3e2fd46b04c720bc6f866")

# 1. The 1-channel model, and a check that it computes what the 3-channel one does on a gray picture.
gray_path = gray_model(nano, WORK / "yolox_nano_gray.onnx")
core, f32 = ov.Core(), {"INFERENCE_PRECISION_HINT": "f32"}
picture = cv2.imread(str(dog), cv2.IMREAD_GRAYSCALE)
x, _ = _letterbox([picture], 416)
three = core.compile_model(str(nano), "CPU", f32)({0: np.repeat(x, 3, 1)})[0]
one_model = core.read_model(str(gray_path)); one_model.reshape([-1, 1, 416, 416])
one = core.compile_model(one_model, "CPU", f32)({0: x})[0]
print(f"1. 1-channel vs 3-channel, float32: largest difference {np.abs(three - one).max():.1e} (float rounding)")

# 2. A panning gray camera: the dog, the bicycle and the car move right; the dog is covered on frames 30-41.
def camera(n=90, covered=range(30, 42), speed=5):
    frames, truth = [], []
    for k in range(n):
        f = np.full((720, 1280), 118, np.uint8)
        x0 = 40 + speed * k
        f[60:60 + picture.shape[0], x0:x0 + picture.shape[1]] = picture
        if k in covered:
            f[60 + 200:60 + 560, x0 + 110:x0 + 350] = 128
        frames.append(f); truth.append(k not in covered)
    return frames, truth

detector = Detector(gray_path, threads=1, classes=("dog", "bicycle", "car"))
tracker = BoxTracker(detector, look_every=3)
frames, truth = camera()
out = [tracker.step({"cam": f})["cam"] for f in frames]
ids = sorted({i for o in out for i in o})
print(f"2. ids ever given: {ids}; on the last frame: " + ", ".join(f"{i}={o.label}" for i, o in sorted(out[-1].items())))

# 3. MCC on the decision a rule would make: "is a dog in view on this frame?", against the truth.
def mcc(truth, said):
    t, s = np.asarray(truth, bool), np.asarray(said, bool)
    tp, tn, fp, fn = (t & s).sum(), (~t & ~s).sum(), (~t & s).sum(), (t & ~s).sum()
    d = np.sqrt(float((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)))
    return (tp * tn - fp * fn) / d if d else 0.0, dict(tp=int(tp), tn=int(tn), fp=int(fp), fn=int(fn))

said = [any(o.label == "dog" for o in f.values()) for f in out]
score, counts = mcc(truth, said)
print(f"3. MCC for 'a dog is in view', look_every=3: {score:.3f} {counts}")
for every in (1, 6):
    t = BoxTracker(detector, look_every=every)
    s = [any(o.label == "dog" for o in t.step({"cam": f})["cam"].values()) for f in frames]
    print(f"   look_every={every}: MCC {mcc(truth, s)[0]:.3f}")

# 4. CPU and memory: every camera in one process with one model, against one process per camera.
cv2.imwrite(str(WORK / "scene.jpg"), picture)
bench = subprocess.run([sys.executable, str(REPO / "bench" / "boxes_cpu.py"), "--model", str(gray_path),
                        "--cameras", "4", "--fps", "10", "--seconds", "15", "--threads", "1", "--image", str(WORK / "scene.jpg")],
                       capture_output=True, text=True, check=True)
for line in bench.stdout.splitlines():
    row = json.loads(line)
    for setup in ("shared", "separate"):
        r = row[setup]
        print(f"4. {row['cameras']} cameras at {row['fps_asked']:g} fps, {setup:8}: {r['processes']} process(es), "
              f"{r['cpu_cores']} CPU cores, {r['memory_mb']} MB, {r['fps_per_camera']} fps each")
